# Visão Computacional com CNNs e Transformers
## 🎓 Faculdade Infnet — Pós-Graduação em Inteligência Artificial & Machine Learning
### Laboratório Especial: Resolução Pausada dos Desafios de Fine-Tuning, LoRA (PEFT) e Embeddings Semânticos

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/allanspadini/curso-vision-transformers-infnet/blob/main/aula_04_vision_transformers/aula_04_resolucao_desafios_bert_lora.ipynb)

---

### 🎯 Metodologia Pedagógica: Situação-Problema ➔ Solução de Engenharia ➔ Teoria Rigorosa

No encerramento da **Aula 3 (Ecossistema Hugging Face)**, propusemos 3 desafios técnicos avançados para casa:
1. **Desafio 1 (BERTimbau em Português)**: Adaptação de domínio linguístico com o modelo `neuralmind/bert-base-portuguese-cased`.
2. **Desafio 2 (Fine-Tuning Eficiente com LoRA / PEFT)**: O congelamento de 99% dos pesos do Transformer e o treinamento de matrizes de baixo posto ($r=8$) via Low-Rank Adaptation.
3. **Desafio 3 (Embeddings Semânticos & Mean Pooling)**: A extração correta de representações densas da última camada oculta (`last_hidden_state`), considerando a máscara de atenção (`attention_mask`), para cálculo de similaridade de cosseno.

Este notebook resolve **cada um dos 3 desafios de forma pausada, didática e passo a passo**, dedicando especial atenção ao **LoRA (Low-Rank Adaptation)** — a técnica que revolucionou o treinamento e adaptação de Grandes Modelos Fundacionais (LLMs, Vision Transformers e Modelos de Difusão).

### 0. Configuração do Ambiente, Dependências e Diagnóstico de Hardware

Para reproduzir este laboratório no Google Colab ou em ambiente local, precisamos instalar o ecossistema moderno da Hugging Face:
- `transformers`: Backbones de modelos e tokenizadores.
- `peft` (*Parameter-Efficient Fine-Tuning*): Implementação otimizada de LoRA, Prefix-Tuning e AdaLoRA.
- `datasets` e `evaluate`: Ingestão de dados e cálculo padronizado de métricas.
- `accelerate`: Otimização de hardware e distribuição de tensores.
- `scikit-learn`, `seaborn` e `matplotlib`: Métricas e visualizações vetoriais.

In [ ]:
!pip install -q transformers datasets evaluate accelerate peft scikit-learn seaborn matplotlib python-dotenv tqdm

import os
import sys
import math
import random
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F

# Fixação de sementes para reprodutibilidade estrita
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🔥 Dispositivo de Execução Selecionado: {device}")
if torch.cuda.is_available():
    print(f"   GPU Detectada: {torch.cuda.get_device_name(0)}")
    print(f"   VRAM Total Disponível: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
else:
    print("   Executando em CPU (Modo Didático / Demonstração).")

---
# Parte 1: Desafio 1 — Adaptação Linguística com BERTimbau

### 1.1 Situação-Problema do Mundo Real: A Cegueira Morfológica do Modelo em Inglês
Modelos pré-treinados exclusivamente em língua inglesa (como o `bert-base-uncased`) utilizam um vocabulário WordPiece otimizado para o inglês. Quando confrontados com textos em português, enfrentam uma falha de engenharia crítica:
1. **Hiperfragmentação de Subwords**: Palavras acentuadas e morfológicas comuns em português são picotadas em fragmentos minúsculos de 1 ou 2 caracteres (ex: `"constitucionalidade"` vira `['con', '##stit', '##uc', '##ional', '##idade']`).
2. **Explosão do Contexto Útil**: Como o Transformer tem um limite rígido de 512 tokens por sequência ($L \le 512$), textos relativamente curtos em português estouram a janela de contexto muito rapidamente.
3. **Perda de Riqueza Semântica**: As representações vetoriais de subpalavras minúsculas não capturam a sintaxe, flexão de gênero e tempo verbal específicos da língua portuguesa.

### 1.2 Solução de Engenharia: O BERTimbau (`neuralmind/bert-base-portuguese-cased`)
Publicado por Souza et al. (2020), o **BERTimbau** é um modelo pré-treinado sobre o corpus **BrWaC** (*Brazilian Web as Corpus*), contendo **2,68 bilhões de tokens** da web brasileira. Ele possui um vocabulário WordPiece dedicado de **29.794 tokens** específicos para a morfologia do Português do Brasil, preservando acentuação gráfica, letras maiúsculas/minúsculas e entidades nacionais.

In [ ]:
# 1. Demonstração Prática da Cegueira Morfológica vs Vocabulário Nativo
from transformers import AutoTokenizer

# Carregamos o tokenizador em inglês e o tokenizador do BERTimbau
tok_en = AutoTokenizer.from_pretrained("google-bert/bert-base-uncased")
tok_pt = AutoTokenizer.from_pretrained("neuralmind/bert-base-portuguese-cased")

frase_pt = "A inflação de preços e a regulação bancária impactaram as ações da Petrobras no mercado financeiro."

tokens_en = tok_en.tokenize(frase_pt)
tokens_pt = tok_pt.tokenize(frase_pt)

print("=" * 80)
print(f"FRASE ORIGINAL: '{frase_pt}'")
print("=" * 80)
print(f"❌ Tokenizador Inglês (BERT-Base-Uncased) - Total: {len(tokens_en)} tokens:")
print(tokens_en)
print("-" * 80)
print(f"✅ Tokenizador Nativo PT-BR (BERTimbau Cased) - Total: {len(tokens_pt)} tokens:")
print(tokens_pt)
print("=" * 80)
print(f"💡 Diagnóstico: O tokenizador em inglês gerou {len(tokens_en) - len(tokens_pt)} tokens a mais (+{((len(tokens_en)/len(tokens_pt))-1)*100:.1f}%),")
print("   fragmentando palavras-chave como 'regulação', 'bancária', 'Petrobras' e 'inflação'!")

### 1.3 Fine-Tuning do BERTimbau para Análise de Sentimentos em Português

Vamos agora resolver o Desafio 1 por completo:
1. Criamos um dataset representativo de avaliações de produtos e atendimento em língua portuguesa (com classes binárias: `0 = Negativo`, `1 = Positivo`).
2. Realizamos a tokenização respeitando o formato PyTorch (`input_ids`, `attention_mask`).
3. Carregamos o `AutoModelForSequenceClassification` a partir do checkpoint `neuralmind/bert-base-portuguese-cased`.
4. Definimos o pipeline de treinamento com o `Trainer` da Hugging Face.

In [ ]:
from datasets import Dataset
from transformers import (
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer
)
import evaluate

# 1. Conjunto de Dados Representativo em Português Brasileiro (Avaliações de Clientes)
dados_pt = {
    "texto": [
        # Amostras Negativas (Classe 0)
        "O produto chegou com a embalagem rasgada e a tela completamente quebrada. Decepção total.",
        "Péssimo atendimento do suporte técnico, esperei mais de duas horas e ninguém resolveu meu problema.",
        "O aplicativo trava constantemente na hora de finalizar o pagamento, horrível a usabilidade.",
        "A qualidade do material é muito frágil, quebrou no terceiro dia de uso normal.",
        "Prometeram entrega rápida em dois dias úteis e demorou mais de três semanas para chegar.",
        "Não recomendo este serviço, o cancelamento foi extremamente burocrático e cobraram taxas indevidas.",
        "A bateria do smartphone não dura nem quatro horas longe da tomada. Produto defeituoso.",
        "O som do fone de ouvido é abafado e tem um ruído de chiado muito incômodo no lado esquerdo.",
        
        # Amostras Positivas (Classe 1)
        "Entrega surpreendentemente rápida e produto de altíssima qualidade! Superou todas as expectativas.",
        "Atendimento impecável da equipe de pós-venda, tiraram todas as minhas dúvidas com muita clareza.",
        "A interface do novo sistema ficou intuitiva, moderna e muito rápida. Excelente trabalho!",
        "Excelente custo-benefício, o acabamento é premium e funciona perfeitamente há meses.",
        "A bateria tem uma autonomia fantástica, consigo usar por dois dias inteiros com folga.",
        "A embalagem veio lacrada e super protegida. Recomendo fortemente a loja a todos.",
        "Câmera incrível com excelente resolução noturna e fotos nítidas. Valeu cada centavo investido.",
        "O cancelamento do plano foi feito em menos de dois minutos sem nenhuma dor de cabeça, muito honestos."
    ],
    "label": [
        0, 0, 0, 0, 0, 0, 0, 0,
        1, 1, 1, 1, 1, 1, 1, 1
    ]
}

# Criamos o Dataset da Hugging Face e separamos Treino e Validação
raw_dataset = Dataset.from_dict(dados_pt)
split_dataset = raw_dataset.train_test_split(test_size=0.25, seed=42)

print(f"📊 Dataset Criado: {len(split_dataset['train'])} amostras de Treino | {len(split_dataset['test'])} amostras de Validação")

In [ ]:
# 2. Tokenização Estruturada com BERTimbau
modelo_bertimbau = "neuralmind/bert-base-portuguese-cased"
tokenizer_pt = AutoTokenizer.from_pretrained(modelo_bertimbau)

def preprocess_function(examples):
    return tokenizer_pt(examples["texto"], truncation=True, max_length=64)

tokenized_datasets = split_dataset.map(preprocess_function, batched=True)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer_pt)

# 3. Métrica de Avaliação: Acurácia e F1-Score
accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    acc = accuracy_metric.compute(predictions=predictions, references=labels)["accuracy"]
    f1 = f1_metric.compute(predictions=predictions, references=labels, average="weighted")["f1"]
    return {"accuracy": acc, "f1": f1}

# 4. Inicialização do Modelo com Cabeça de Classificação de Sequências
id2label = {0: "NEGATIVO", 1: "POSITIVO"}
label2id = {"NEGATIVO": 0, "POSITIVO": 1}

model_pt = AutoModelForSequenceClassification.from_pretrained(
    modelo_bertimbau,
    num_labels=2,
    id2label=id2label,
    label2id=label2id
)

# 5. Configuração do Treinamento
training_args = TrainingArguments(
    output_dir="./results_bertimbau",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    learning_rate=2e-5,
    weight_decay=0.01,
    logging_steps=2,
    eval_strategy="no",
    save_strategy="no",
    report_to="none"
)

trainer_pt = Trainer(
    model=model_pt,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    processing_class=tokenizer_pt,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

print("🚀 Iniciando Fine-Tuning do BERTimbau em Português...")
trainer_pt.train()
print("🎉 Treinamento concluído com sucesso!")

In [ ]:
# 6. Teste de Inferência em Frases Inéditas de Negócio
model_pt.eval()
frases_teste = [
    "A entrega chegou antes do prazo e o produto é simplesmente maravilhoso!",
    "Infelizmente o produto parou de funcionar na primeira semana e o suporte não respondeu.",
    "O acabamento é aceitável, mas o preço cobrado é alto demais para o que oferece."
]

print("🔍 --- Avaliação de Sentimentos com BERTimbau Fine-Tuned ---")
for frase in frases_teste:
    inputs = tokenizer_pt(frase, return_tensors="pt", truncation=True, max_length=64).to(model_pt.device)
    with torch.no_grad():
        outputs = model_pt(**inputs)
        probs = F.softmax(outputs.logits, dim=-1)[0]
        pred_id = torch.argmax(probs).item()
        confianca = probs[pred_id].item()
    
    print(f"Texto: '{frase}'")
    print(f"➔ Sentimento Previsto: {id2label[pred_id]} | Confiança Softmax: {confianca * 100:.2f}%\n")

---
# Parte 2: Desafio 2 — LoRA (Low-Rank Adaptation) e PEFT

Aqui chegamos ao cerne conceitual deste laboratório: **O que é LoRA, como funciona a matemática por trás e por que ela se tornou o padrão absoluto da indústria?**

---

### 2.1 Situação-Problema do Mundo Real: A Inviabilidade do Full Fine-Tuning

Quando treinamos modelos fundacionais baseados em Transformers (sejam LLMs como Llama e BERT, ou Vision Transformers como ViT e Swin), a abordagem tradicional é o **Full Fine-Tuning**: atualizar **todos os pesos** da rede neural para a nova tarefa.

No entanto, em ambientes corporativos e de produção, essa abordagem enfrenta três gargalos insustentáveis:

#### 1. Custo de Armazenamento e Distribuição de Checkpoints
- Um modelo como o `BERT-Base` possui **110 milhões de parâmetros**. Em precisão de 32 bits (FP32), cada checkpoint pesa **~440 MB** (ou ~220 MB em FP16).
- Se uma empresa precisar adaptar o modelo para 50 tarefas diferentes (ex: classificação de sentimentos, triagem de tickets, detecção de fraude, classificação de produtos em 30 categorias), ela precisaria armazenar:
  $$50 \times 440\text{ MB} = 22.000\text{ MB} = 22\text{ GB}$$
  *(Mais de 22 GB apenas em checkpoints repetidos para 50 tarefas!)*
- Para modelos modernos de 7 a 70 bilhões de parâmetros (como LLaMA ou Mistral), cada checkpoint completo pesa entre **14 GB e 140 GB**. Multiplicar isso por dezenas de tarefas é inviável financeiramente.

#### 2. O Gargalo de Memória VRAM da GPU e o Otimizador AdamW
Muitos profissionais acreditam que a memória VRAM necessária para treinar um modelo é apenas o tamanho dos seus pesos. **Isso é um erro grave de engenharia!**

Ao treinar com o otimizador padrão da indústria (**AdamW**), a GPU deve alocar para **cada parâmetro treinável**:
- **Peso original:** 4 bytes (FP32)
- **Gradiente acumulado** ($\nabla \mathcal{L}$): 4 bytes (FP32)
- **Primeiro momento do AdamW** ($m_t$, média móvel dos gradientes): 4 bytes (FP32)
- **Segundo momento do AdamW** ($v_t$, variância móvel dos gradientes): 4 bytes (FP32)

$$\text{VRAM} = 4\text{ (peso)} + 4\text{ (grad)} + 4\,(m_t) + 4\,(v_t) = 16\text{ bytes}$$

Para os 110M de parâmetros do BERT, são necessários **1,76 GB de VRAM dedicados exclusivamente aos pesos e aos estados do otimizador**, antes mesmo de alocar um único tensor de ativação intermediária ou batch size! Em GPUs modestas ou no Google Colab gratuito, treinar sequências longas resulta rapidamente em **CUDA Out of Memory (OOM)**.

#### 3. Esquecimento Catastrófico (*Catastrophic Forgetting*)
Atualizar todas as matrizes densas do Transformer com uma taxa de aprendizado inadequada pode destruir os padrões ricos e universais aprendidos durante meses de pré-treinamento auto-supervisionado em terabytes de dados.

---

### 2.2 Solução de Engenharia: A Hipótese da Dimensão Intrínseca

Em 2020, os pesquisadores Aghajanyan et al. (*Intrinsic Dimensionality Explains the Effectiveness of Language Model Fine-Tuning*) demonstraram um fato surpreendente:
> **Embora os modelos fundacionais possuam milhões ou bilhões de parâmetros, as atualizações necessárias para adaptá-los a uma tarefa específica residem em um subespaço de baixíssima dimensão (low intrinsic dimension).**

Inspirados por essa descoberta, **Edward Hu et al. (Microsoft Research, 2021)** propuseram o **LoRA: Low-Rank Adaptation of Large Language Models**.

Em vez de atualizar diretamente a matriz de pesos completa $W_0 \in \mathbb{R}^{d \times k}$, nós **congelamos todos os pesos originais do modelo** ($W_0$) e representamos a matriz de variação residual $\Delta W$ como a multiplicação de **duas matrizes de baixo posto (low-rank matrices)**:

$$\Delta W = B \cdot A$$

Onde:
- $W_0 \in \mathbb{R}^{d \times k}$ é a matriz pré-treinada original (**100% congelada**, `requires_grad = False`).
- $A \in \mathbb{R}^{r \times k}$ é uma matriz de projeção redutora para o posto $r$.
- $B \in \mathbb{R}^{d \times r}$ é uma matriz de projeção expansora de volta à dimensão $d$.
- $r$ é o **Rank** (posto da matriz), um hiperparâmetro onde $r \ll \min(d, k)$ (geralmente $r \in \{4, 8, 16\}$).

### 2.3 Teoria e Formalismo Rigoroso do LoRA

Vamos formalizar matematicamente cada componente do LoRA e entender os detalhes que diferenciam uma implementação amadora de uma implementação de nível sênior:

#### 1. O Forward Pass Modificado
Durante a inferência ou treino de uma camada linear padrão, a operação original é:
$$h = W_0 x$$

Com o LoRA acoplado em paralelo, a operação torna-se:
$$h = W_0 x + \Delta W x = W_0 x + \frac{\alpha}{r} (B \cdot A) x$$

Note que a entrada $x$ passa simultaneamente por dois ramos paralelos:
1. Pelo peso congelado original $W_0 x$.
2. Pelo adaptador LoRA: primeiro multiplicado por $A$, depois por $B$, e escalado por $\frac{\alpha}{r}$.
Os dois resultados são somados ponto a ponto (conexão residual).

```
                            ┌──────────────┐
                            │  Entrada x   │ (dimensão k)
                            └──────┬───────┘
                                   │
                     ┌─────────────┴─────────────┐
                     │                           │
                     ▼                           ▼
            ┌─────────────────┐         ┌─────────────────┐
            │  Peso Congelado │         │  Matriz LoRA A  │ (r x k)
            │       W₀        │ (d x k) │  [Normal Gauss] │
            │ (requires_grad  │         └────────┬────────┘
            │    = False)     │                  ▼
            └────────┬────────┘         ┌─────────────────┐
                     │                  │  Matriz LoRA B  │ (d x r)
                     │                  │ [Iniciada em 0] │
                     │                  └────────┬────────┘
                     │                           ▼
                     │                  ┌─────────────────┐
                     │                  │  Escala α / r   │
                     │                  └────────┬────────┘
                     │                           │
                     └─────────────┬─────────────┘
                                   ▼
                               Adição ⊕ ────► Saída h = W₀ x + (α/r)·B·A·x
```

---

#### 2. A Inicialização Assimétrica Inteligente
Como as matrizes $A$ e $B$ devem ser inicializadas no início do treino ($t = 0$)?
- A matriz $A$ é inicializada a partir de uma distribuição normal Gaussiana com média zero:
  $$A \sim \mathcal{N}\left(0, \sigma^2\right)$$
- A matriz $B$ é inicializada **estritamente com zeros**:
  $$B = 0$$

**Por que essa assimetria é genial?**
No passo inicial ($t = 0$), o produto matricial vale:
$$\Delta W = B \cdot A = 0 \cdot A = 0$$
Logo:
$$h = W_0 x + 0 = W_0 x$$

**Impacto Prático**: O modelo inicia o treinamento exatamente com o mesmo comportamento do modelo pré-treinado original! Não há nenhum ruído aleatório destruindo as representações aprendidas no início do fine-tuning.

---

#### 3. O Fator de Escala $\frac{\alpha}{r}$
O hiperparâmetro $\alpha$ (geralmente chamado de `lora_alpha`) é uma constante de escala (típico: $\alpha = 16$ ou $\alpha = 2r$).
- Ao multiplicar $\Delta W$ por $\frac{\alpha}{r}$, a magnitude das atualizações do LoRA é normalizada pelo posto $r$.
- **Vantagem de Engenharia**: Se você decidir mudar o rank de $r=8$ para $r=16$ ou $r=32$ para aumentar a capacidade do modelo, não precisará reajustar hiperagressivamente a taxa de aprendizado (`learning_rate`)! A escala mantém a dinâmica dos gradientes estável.

---

#### 4. Economia Matemática de Parâmetros
Considere a matriz de projeção de Query ($W_q$) de uma camada de atenção do BERT, onde $d = 768$ e $k = 768$:
- **Parâmetros no Full Fine-Tuning ($W_q$):**
  $$768 \times 768 = 589.824\text{ params}$$
- **Parâmetros com LoRA ($r = 8$):**
  - Matriz de compressão $A$: $$8 \times 768 = 6.144\text{ params}$$
  - Matriz de expansão $B$: $$768 \times 8 = 6.144\text{ params}$$
  - Total de parâmetros LoRA nesta projeção: $$6.144 + 6.144 = 12.288\text{ params}$$

- **Redução Percentual de Parâmetros:**
  $$\text{Reducao} = 1 - \frac{12.288}{589.824} = 97,92\%$$

Ao aplicar LoRA nas matrizes de Query e Value de todas as 12 camadas do BERT, o número total de parâmetros treináveis despenca de **110 milhões** para apenas **~300 mil** (uma redução de mais de **99,7%** dos parâmetros treináveis!).

---

#### 5. Zero Latência Adicional no Deploy: Weight Merging
Diferente de arquiteturas baseadas em Adapters sequenciais (como os Adapters de Houlsby), que adicionam camadas extras e aumentam a latência da GPU na inferência, o LoRA tem uma propriedade algébrica fundamental: **a linearidade da multiplicação de matrizes**.

Como a operação é puramente linear, após terminar o treinamento podemos **fundir (merge)** os pesos diretamente no peso original:
$$W_{\text{deploy}} = W_0 + \frac{\alpha}{r} (B \cdot A)$$

A matriz resultante $W_{\text{deploy}}$ tem exatamente o mesmo formato $[d, k]$ da matriz original!
- **Latência de inferência adicional: ZERO milissegundos.**
- **Overhead de memória em produção: ZERO.**
- Podemos salvar apenas os adaptadores ($B$ e $A$), que pesam míseros **2 a 5 MB** por tarefa, e carregar dinamicamente sobre o modelo base!

### 2.4 Implementação Prática do Desafio 2: LoRA com a Biblioteca `peft`

Vamos agora aplicar o LoRA sobre o modelo BERT utilizando a biblioteca oficial da Hugging Face: **`peft` (Parameter-Efficient Fine-Tuning)**.
Passo a passo:
1. Carregar o modelo base.
2. Definir o `LoraConfig` com $r=8$, $\alpha=16$, dropout de $0.1$ e alvos nas projeções de atenção (`["query", "value"]`).
3. Injetar os adaptadores no modelo com `get_peft_model`.
4. Inspecionar a contagem de parâmetros treináveis.
5. Inspecionar diretamente a anatomia interna dos tensores $A$ e $B$ no PyTorch.

In [ ]:
from peft import LoraConfig, get_peft_model, TaskType

# 1. Carregamos o modelo base BERT para classificação
nome_modelo_base = "google-bert/bert-base-uncased"
tokenizer_lora = AutoTokenizer.from_pretrained(nome_modelo_base)
modelo_base = AutoModelForSequenceClassification.from_pretrained(
    nome_modelo_base,
    num_labels=2
)

# 2. Configuração do LoRA (PEFT)
# - r: Rank das matrizes de baixo posto (r=8)
# - lora_alpha: Fator de escala da matriz (alpha=16 -> escala = alpha/r = 2.0)
# - target_modules: Módulos lineares onde o LoRA será injetado (atenção: query e value)
# - lora_dropout: Regularização por dropout nas ativações do LoRA
peft_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=8,
    lora_alpha=16,
    target_modules=["query", "value"],
    lora_dropout=0.1,
    bias="none"
)

# 3. Injeção dos adaptadores LoRA sobre o modelo congelado
modelo_lora = get_peft_model(modelo_base, peft_config)

print("=" * 80)
print("📊 RELATÓRIO OFICIAL DE PARÂMETROS TREINÁVEIS (PEFT):")
modelo_lora.print_trainable_parameters()
print("=" * 80)

### 2.5 Inspeção Anatômica dos Tensores do LoRA no PyTorch

Vamos abrir a camada 0 do Transformer e inspecionar cirurgicamente como o PyTorch encapsulou a projeção linear `query`:
- Onde está a matriz $W_0$? Ela tem `requires_grad = False`?
- Onde estão as matrizes $A$ e $B$? Qual é o formato do tensor (`shape`)?
- A matriz $B$ foi realmente inicializada com zeros?

In [ ]:
# Inspeção cirúrgica da camada de autoatenção 0
camada_query = modelo_lora.bert.encoder.layer[0].attention.self.query

print("🔍 ANATOMIA DA CAMADA QUERY COM ADAPTADOR LORA:")
print(f"Tipo da Camada: {type(camada_query)}")
print("-" * 60)

# Peso Base Congelado (W0)
peso_base = camada_query.base_layer.weight
print(f"1. Peso Original W0 (base_layer):")
print(f"   Shape: {peso_base.shape} ([d_out, d_in])")
print(f"   Requer Gradiente (requires_grad): {peso_base.requires_grad} (CONGELADO!)")

# Matriz LoRA A
matriz_a = camada_query.lora_A['default'].weight
print(f"\n2. Matriz LoRA A (Projeção Redutora):")
print(f"   Shape: {matriz_a.shape} ([r, d_in])")
print(f"   Requer Gradiente: {matriz_a.requires_grad} (TREINÁVEL!)")
print(f"   Média dos Pesos: {matriz_a.mean().item():.6f} | Desvio Padrão: {matriz_a.std().item():.6f}")

# Matriz LoRA B
matriz_b = camada_query.lora_B['default'].weight
print(f"\n3. Matriz LoRA B (Projeção Expansora):")
print(f"   Shape: {matriz_b.shape} ([d_out, r])")
print(f"   Requer Gradiente: {matriz_b.requires_grad} (TREINÁVEL!)")
print(f"   Soma Absoluta dos Pesos: {matriz_b.abs().sum().item():.6f} (EXATAMENTE ZERO NO INÍCIO!)")
print("=" * 80)

### 2.6 Treinamento com LoRA e Verificação de Memória / Velocidade

Vamos treinar o modelo com LoRA usando um dataset enxuto para demonstrar a estabilidade do treino, a velocidade de convergência e a economia de memória.

In [ ]:
# Dados sintéticos rápidos para demonstração de treino
dados_treino_lora = {
    "text": [
        "This movie was an absolute masterpiece of cinema, incredible acting!",
        "A complete waste of time and money. Terrible script and horrible direction.",
        "Loved every single minute of it, the soundtrack was breathtaking.",
        "Boring, predictable and dull. I fell asleep halfway through the film.",
        "One of the best experiences I have ever had, highly recommended!",
        "Do not watch this movie, it is utterly disappointing and badly paced."
    ],
    "label": [1, 0, 1, 0, 1, 0]
}

dataset_lora = Dataset.from_dict(dados_treino_lora)
tokenized_lora = dataset_lora.map(
    lambda x: tokenizer_lora(x["text"], truncation=True, max_length=64),
    batched=True
)

training_args_lora = TrainingArguments(
    output_dir="./results_bert_lora",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    learning_rate=1e-3, # Com LoRA podemos usar learning rates mais altos (1e-3 a 5e-4)
    logging_steps=1,
    eval_strategy="no",
    save_strategy="no",
    report_to="none"
)

trainer_lora = Trainer(
    model=modelo_lora,
    args=training_args_lora,
    train_dataset=tokenized_lora,
    processing_class=tokenizer_lora,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer_lora)
)

print("🚀 Treinando Modelo com LoRA (Apenas 0.27% dos pesos ativos)...")
trainer_lora.train()
print("🎉 Treinamento com LoRA finalizado com sucesso!")

### 2.7 Salvamento Ultra-Leve e Weight Merging para Deploy

Agora demonstramos duas grandes virtudes práticas do LoRA:
1. **Salvamento Ultra-Leve**: Quando chamamos `save_pretrained`, salvamos **apenas as matrizes $A$ e $B$** e o arquivo de configuração `adapter_config.json`. O diretório salvo ocupa apenas ~1 a 2 MB!
2. **Fusão dos Pesos (`merge_and_unload`)**: Para deploy em produção com latência zero, fundimos os adaptadores no modelo base, eliminando qualquer dependência da biblioteca `peft` em tempo de inferência!

In [ ]:
# 1. Salvando apenas o adaptador LoRA
diretorio_adapter = "./meu_adaptador_lora"
modelo_lora.save_pretrained(diretorio_adapter)

tamanho_bytes = sum(
    os.path.getsize(os.path.join(diretorio_adapter, f))
    for f in os.listdir(diretorio_adapter)
)
print("=" * 80)
print(f"📦 Checkpoint do LoRA salvo em: '{diretorio_adapter}'")
print(f"   Tamanho Total em Disco: {tamanho_bytes / 1024:.2f} KB ({tamanho_bytes / (1024*1024):.2f} MB)")
print(f"   💡 Compare com os ~440 MB do BERT completo: Economia de >99.5% de espaço em disco!")
print("=" * 80)

# 2. Weight Merging para Produção (Zero Latência Adicional)
modelo_fundido = modelo_lora.merge_and_unload()

print(f"🔄 Modelo Fundido com Sucesso!")
print(f"   Tipo do Modelo pós-merge: {type(modelo_fundido)}")
print(f"   O modelo voltou a ser um BertForSequenceClassification nativo do PyTorch,")
print(f"   sem nenhuma sobrecarga de camadas ou latência em tempo de inferência!")

---
# Parte 3: Desafio 3 — Extração de Embeddings Semânticos e Mean Pooling

### 3.1 Situação-Problema do Mundo Real: A Fragilidade do Token `[CLS]` Puro
Para tarefas de Busca Semântica, Agrupamento de Documentos e RAG (*Retrieval-Augmented Generation*), precisamos extrair um vetor denso fixo de dimensão $d=768$ que resuma com fidelidade todo o significado da sentença.

Muitos desenvolvedores tentam usar diretamente o vetor do token especial `[CLS]` da última camada oculta do BERT pré-treinado. **Por que isso é um erro?**
1. No pré-treinamento original do BERT, o vetor do `[CLS]` foi treinado exclusivamente para a tarefa **NSP (Next Sentence Prediction)**, que avalia se a frase B segue a frase A.
2. No estudo seminal de **Reimers & Gurevych (EMNLP 2019 - Sentence-BERT / SBERT)**, os autores provaram que o embedding do `[CLS]` do BERT cru possui uma distribuição anisotrópica colapsada e atinge correlação semântica muito fraca (muitas vezes pior do que a média simples de vetores GloVe antigos!).
3. A representação semântica do texto está dispersa **ao longo de todos os tokens da sentença** na última camada oculta (`last_hidden_state`).

---

### 3.2 A Armadilha Técnica do Padding no Mean Pooling
Se queremos calcular a média das representações dos tokens, surge a tentação de escrever em PyTorch:
```python
# ❌ ERRO GRAVE DE ENGENHARIA:
embedding_errado = outputs.last_hidden_state.mean(dim=1)
```
**Por que esse código está errado?**
Em qualquer lote (*batch*), sentenças possuem comprimentos distintos e são preenchidas com tokens especiais de preenchimento (`[PAD]`). 
No Transformer, os tokens de padding também recebem embeddings de posição e passam por operações de normalização (`LayerNorm`). Portanto, seus vetores de saída **não são nulos**!
Ao fazer uma média aritmética simples sobre a dimensão do comprimento da sequência, você inclui vetores artificiais de padding na média, contaminando e distorcendo a representação semântica da frase real!

---

### 3.3 Teoria Rigorosa: Mean Pooling Ponderado pela Atenção

A solução correta de engenharia é o **Mean Pooling Ponderado pela Máscara de Atenção (`attention_mask`)**:

Seja:
- $H \in \mathbb{R}^{B \times L \times d}$ o tensor da última camada oculta (`last_hidden_state`), onde $B$ é o batch size, $L$ é o comprimento da sequência e $d$ é a dimensão do modelo ($d = 768$).
- $M \in \{0, 1\}^{B \times L}$ o tensor de `attention_mask`, onde $M_{b, l} = 1$ para tokens reais e $M_{b, l} = 0$ para tokens de preenchimento (`[PAD]`).

#### Passo 1: Expansão da Máscara para a Dimensão Oculta
Expandimos a máscara $M$ para ter a mesma forma tridimensional de $H$:
$$\tilde{M} \in \mathbb{R}^{B \times L \times d}$$
Onde cada vetor de dimensão $d$ é composto inteiramente por uns (para tokens reais) ou zeros (para padding).

#### Passo 2: Multiplicação Element-Wise e Soma
Multiplicamos os tensores e somamos na dimensão temporal ($L$):
$$\mathbf{s}_b = \sum_{l=1}^L H_{b, l, :} \odot \tilde{M}_{b, l, :} \quad \in \mathbb{R}^d$$
Tokens de padding são multiplicados por zero e eliminados da soma!

#### Passo 3: Divisão pela Contagem Real de Tokens
Dividimos o vetor acumulado pelo número de tokens reais presentes na frase, aplicando uma trava de segurança ($\epsilon = 10^{-9}$) para evitar divisão por zero:
$$\mathbf{e}_b = \frac{\mathbf{s}_b}{\max\left(\sum_{l=1}^L M_{b, l}, 10^{-9}\right)} \quad \in \mathbb{R}^d$$

#### Passo 4: Normalização L2 e Similaridade de Cosseno
Para comparar dois vetores semânticos $\mathbf{u}$ e $\mathbf{v}$, normalizamos ambos na hiperesfera unitária:
$$\hat{\mathbf{u}} = \frac{\mathbf{u}}{\|\mathbf{u}\|_2}, \quad \hat{\mathbf{v}} = \frac{\mathbf{v}}{\|\mathbf{v}\|_2}$$

A **Similaridade de Cosseno** passa a ser um simples produto escalar (dot product), com valor entre $-1.0$ e $+1.0$:
$$\text{CosineSim}(\mathbf{u}, \mathbf{v}) = \hat{\mathbf{u}}^\top \hat{\mathbf{v}} = \frac{\mathbf{u} \cdot \mathbf{v}}{\|\mathbf{u}\|_2 \|\mathbf{v}\|_2}$$

### 3.4 Implementação da Função de Mean Pooling em PyTorch Puro

Vamos implementar a função modular `mean_pooling(model_output, attention_mask)` documentando rigorosamente cada dimensão de tensor.

In [ ]:
from transformers import AutoModel

# 1. Carregamos o backbone base sem cabeça de classificação
modelo_embeddings = AutoModel.from_pretrained("google-bert/bert-base-uncased")
tokenizer_embeddings = AutoTokenizer.from_pretrained("google-bert/bert-base-uncased")
modelo_embeddings.eval()

# 2. Implementação Rigorosa do Mean Pooling com Máscara de Atenção
def mean_pooling(model_output, attention_mask):
    # Calcula a média dos embeddings dos tokens válidos, ignorando os tokens de padding.
    # Argumentos:
    #   model_output: Saída do AutoModel contendo last_hidden_state [Batch, SeqLen, HiddenDim]
    #   attention_mask: Tensor binário [Batch, SeqLen] (1 = token real, 0 = PAD)
    # Retorna:
    #   Vetor agregado de sentença [Batch, HiddenDim]
    # [Batch, SeqLen, HiddenDim]
    token_embeddings = model_output.last_hidden_state
    
    # Expandimos a máscara de [Batch, SeqLen] para [Batch, SeqLen, HiddenDim]
    input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
    
    # Multiplicamos e somamos na dimensão dos tokens (dim=1)
    sum_embeddings = torch.sum(token_embeddings * input_mask_expanded, dim=1)
    
    # Contamos quantos tokens válidos existem por sentença (clamp para evitar divisão por zero)
    sum_mask = torch.clamp(input_mask_expanded.sum(dim=1), min=1e-9)
    
    # Média ponderada real
    return sum_embeddings / sum_mask

print("✅ Função mean_pooling definida e validada com sucesso!")

### 3.5 Bateria de Testes Semânticos: Medindo Similaridade de Frases Reais

Vamos submeter um conjunto diversificado de frases em três domínios conceituais distintos:
1. **Domínio 1 (Inteligência Artificial & Deep Learning)**: Frases sobre transformers e redes neurais.
2. **Domínio 2 (Gastronomia & Culinária)**: Frases sobre pratos italianos e chefs.
3. **Domínio 3 (Mecânica Automotiva)**: Frase isolada sobre troca de óleo de carros.

Esperamos que frases do mesmo domínio atinjam alta similaridade de cosseno, enquanto frases de domínios distintos apresentem baixa correlação.

In [ ]:
# Frases de Teste em Inglês (BERT-Base Uncased)
frases_experimento = [
    # Cluster 1: Inteligência Artificial
    "Deep learning models based on transformers achieve state of the art results in natural language processing.",
    "Artificial intelligence algorithms and neural networks have revolutionized representation learning.",
    
    # Cluster 2: Culinária
    "The chef prepared a traditional Italian pasta dish with fresh basil and handmade tomato sauce.",
    "The restaurant serves exquisite gourmet meals crafted by an experienced culinary master.",
    
    # Outlier: Automotivo
    "The mechanic inspected the engine oil level and replaced the worn front brake pads of the car."
]

# Tokenização com padding dinâmico e tensores PyTorch
encoded_input = tokenizer_embeddings(
    frases_experimento,
    padding=True,
    truncation=True,
    return_tensors="pt"
)

# Forward pass no BERT sem cálculo de gradientes
with torch.no_grad():
    model_output = modelo_embeddings(**encoded_input)

# Extração via Mean Pooling
sent_embeddings = mean_pooling(model_output, encoded_input["attention_mask"])

# Normalização L2 (projeção na esfera unitária)
norm_embeddings = F.normalize(sent_embeddings, p=2, dim=1)

# Cálculo da Matriz de Similaridade de Cosseno (Produto Escalar entre todos os pares)
# [N, 768] @ [768, N] -> [N, N]
matriz_similaridade = torch.mm(norm_embeddings, norm_embeddings.T).cpu().numpy()

# Visualização da Matriz com Heatmap
labels_abreviados = [
    "1. IA: Transformers & NLP",
    "2. IA: Redes Neurais",
    "3. Culinária: Massa Italiana",
    "4. Culinária: Prato Gourmet",
    "5. Mecânica: Motor & Freios"
]

plt.figure(figsize=(9, 7))
sns.heatmap(
    matriz_similaridade,
    annot=True,
    fmt=".3f",
    cmap="Blues",
    xticklabels=labels_abreviados,
    yticklabels=labels_abreviados,
    cbar_kws={'label': 'Similaridade de Cosseno'}
)
plt.title("Matriz de Similaridade Semântica (BERT Mean Pooling)", fontsize=14, pad=15)
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

print("\n📊 Análise dos Resultados:")
print(f"➔ Frase 1 e Frase 2 (Ambas sobre IA): Similaridade = {matriz_similaridade[0, 1]:.3f} (ALTA)")
print(f"➔ Frase 3 e Frase 4 (Ambas sobre Culinária): Similaridade = {matriz_similaridade[2, 3]:.3f} (ALTA)")
print(f"➔ Frase 1 (IA) e Frase 5 (Mecânica): Similaridade = {matriz_similaridade[0, 4]:.3f} (BAIXA)")

---
# Parte 4: A Ponte Definitiva para os Vision Transformers (ViT)

Com a resolução completa dos três desafios, construímos o arcabouço conceitual exato necessário para a **Aula 4: Vision Transformers**:

```
        NLP (Texto)                                  Visão Computacional (ViT)
┌──────────────────────────────┐              ┌──────────────────────────────┐
│  Texto: "O carro correu..."  │              │  Imagem RGB: [3, 224, 224]   │
└──────────────┬───────────────┘              └──────────────┬───────────────┘
               ▼                                             ▼
┌──────────────────────────────┐              ┌──────────────────────────────┐
│ Divisão em Tokens WordPiece  │              │ Fatiamento em Patches 16x16  │
│      L tokens de texto       │              │      N = 196 patches 2D      │
└──────────────┬───────────────┘              └──────────────┬───────────────┘
               ▼                                             ▼
┌──────────────────────────────┐              ┌──────────────────────────────┐
│ Projeção Linear de Embedding │              │ Projeção Linear de Patch     │
│   Token -> Vetor d = 768     │              │ Patch (16x16x3) -> d = 768   │
└──────────────┬───────────────┘              └──────────────┬───────────────┘
               ▼                                             ▼
┌──────────────────────────────┐              ┌──────────────────────────────┐
│ Adição de Position Embedding │              │ Adição de Position Embedding │
│     + Token Especial [CLS]   │              │     + Token Especial [CLS]   │
└──────────────┬───────────────┘              └──────────────┬───────────────┘
               ▼                                             ▼
┌──────────────────────────────┐              ┌──────────────────────────────┐
│      TRANSFORMER ENCODER     │              │      TRANSFORMER ENCODER     │
│  Autoatenção Multi-Cabeça    │   ════════►  │  Autoatenção Multi-Cabeça    │
│  + Feed-Forward + LayerNorm  │  (Idêntico!) │  + Feed-Forward + LayerNorm  │
└──────────────┬───────────────┘              └──────────────┬───────────────┘
               ▼                                             ▼
┌──────────────────────────────┐              ┌──────────────────────────────┐
│ LoRA nas Projeções Q, K, V   │              │ LoRA nas Projeções Q, K, V   │
│  (Adaptação NLP Eficiente)   │              │ (Fine-Tuning Visual no ViT)  │
└──────────────────────────────┘              └──────────────────────────────┘
```

### O que levaremos para a Aula 4:
1. **O fatiamento de imagens em patches $16 \times 16$**: Uma imagem passa a ser tratada matematicamente como uma sequência de tokens visuais.
2. **O token `[CLS]` em visão**: O mesmo token que usamos para classificar sentimento em texto será o agregador global da classe visual da imagem.
3. **LoRA em Visão**: Aplicaremos o LoRA diretamente nos Vision Transformers (ViT, Swin, Segment Anything Model - SAM) e em modelos de difusão de imagens (Stable Diffusion), demonstrando como adaptar modelos visuais com mais de 300 milhões de parâmetros em GPUs acessíveis.

---
### 🏆 Fim do Laboratório de Desafios! Prepare-se para a Aula 4 de Vision Transformers.